# FactChecker AI - God-Level DeBERTa Fine-Tuning

Runtime: T4 GPU required - Runtime > Change runtime type > T4 GPU

Steps:
1. Loads 10 diverse fake-news datasets
2. Fine-tunes microsoft/deberta-v3-base
3. Evaluates on hard LIAR benchmark
4. Pushes model to HuggingFace Hub

Before running: add HF_TOKEN to Colab secrets (key icon on left sidebar)

In [ ]:
%pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy huggingface_hub sentencepiece protobuf
print('Done')

In [ ]:
import torch
from huggingface_hub import login
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print('HuggingFace login OK')
except Exception as e:
    print(f'HF login failed: {e} - model will save locally only')
    HF_TOKEN = None

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

HF_USERNAME = 'your-hf-username'  # CHANGE THIS
MODEL_NAME  = f'{HF_USERNAME}/factchecker-deberta'
print(f'Will push to: {MODEL_NAME}')

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

frames = []

def add(df, source):
    df = df[['text', 'label']].copy()
    df['source'] = source
    frames.append(df)
    print(f'  {source}: {len(df):,} rows')

# 1. clmentbisaillon (44k)
try:
    ds = load_dataset('clmentbisaillon/fake-and-real-news-dataset', split='train')
    df = ds.to_pandas()
    df['text'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).str.strip()
    df['label'] = df['label'].astype(int)
    add(df, 'clmentbisaillon')
except Exception as e: print(f'  skip clmentbisaillon: {e}')

# 2. ErfanMoosaviMonazzah (44k)
try:
    ds = load_dataset('ErfanMoosaviMonazzah/fake-news-detection-dataset-English', split='train')
    df = ds.to_pandas()
    df['text'] = df['text'].fillna('').str.strip()
    df['label'] = df['label'].astype(int)
    add(df, 'erfan')
except Exception as e: print(f'  skip erfan: {e}')

# 3. LIAR (12.8k PolitiFact)
try:
    for split in ['train', 'validation', 'test']:
        ds = load_dataset('ucsbnlp/liar', split=split)
        df = ds.to_pandas()
        df['text'] = df['statement'].fillna('').str.strip()
        df['label'] = df['label'].map({0:1, 1:1, 2:None, 3:None, 4:0, 5:0})
        df = df.dropna(subset=['label'])
        df['label'] = df['label'].astype(int)
        add(df, f'liar_{split}')
except Exception as e: print(f'  skip liar: {e}')

# 4. COVID-19
try:
    ds = load_dataset('nanyy1025/covid19_fake_news', split='train')
    df = ds.to_pandas()
    text_col = 'tweet' if 'tweet' in df.columns else 'text' if 'text' in df.columns else df.columns[0]
    label_col = 'label' if 'label' in df.columns else df.columns[-1]
    df['text'] = df[text_col].fillna('').str.strip()
    df['label'] = pd.to_numeric(df[label_col], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    add(df, 'covid19')
except Exception as e: print(f'  skip covid19: {e}')

# 5. Elections
try:
    ds = load_dataset('newsmediabias/fake_news_elections_labelled_data', split='train')
    df = ds.to_pandas()
    df['text'] = df['text'].fillna('').str.strip()
    df['label'] = df['label'].str.strip().str.lower().map({'fake':1,'real':0})
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    add(df, 'elections')
except Exception as e: print(f'  skip elections: {e}')

# 6. DiFraud (sample 20k)
try:
    ds = load_dataset('difraud/difraud', split='train')
    df = ds.to_pandas()
    df['text'] = df['text'].fillna('').str.strip()
    df['label'] = df['label'].astype(int)
    df = df.sample(min(20000, len(df)), random_state=42)
    add(df, 'difraud')
except Exception as e: print(f'  skip difraud: {e}')

# 7. WELFake (sample 25k)
try:
    ds = load_dataset('ai4bharat/WELFake', split='train')
    df = ds.to_pandas()
    t = df.get('title', pd.Series([''] * len(df))).fillna('')
    b = df.get('text', pd.Series([''] * len(df))).fillna('')
    df['text'] = (t + ' ' + b).str.strip()
    df['label'] = df['label'].astype(int)
    df = df.sample(min(25000, len(df)), random_state=42)
    add(df, 'welfake')
except Exception as e: print(f'  skip welfake: {e}')

# 8. GossipCop
try:
    ds = load_dataset('Yakoob-Khan/Fake-News-Detection', split='train')
    df = ds.to_pandas()
    text_col = next((c for c in ['text','title','content'] if c in df.columns), df.columns[0])
    label_col = next((c for c in ['label','fake','class'] if c in df.columns), df.columns[-1])
    df['text'] = df[text_col].fillna('').str.strip()
    df['label'] = pd.to_numeric(df[label_col], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    add(df, 'gossipcop')
except Exception as e: print(f'  skip gossipcop: {e}')

# 9. Politifact
try:
    ds = load_dataset('sherryycxie/politifact', split='train')
    df = ds.to_pandas()
    text_col = next((c for c in ['statement','text','claim'] if c in df.columns), df.columns[0])
    label_col = next((c for c in ['label','verdict','class'] if c in df.columns), df.columns[-1])
    df['text'] = df[text_col].fillna('').str.strip()
    lmap = {'false':1,'pants-fire':1,'barely-true':1,'fake':1,'0':1,'true':0,'mostly-true':0,'half-true':0,'real':0,'1':0}
    df['label'] = df[label_col].astype(str).str.lower().str.strip().map(lmap)
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    add(df, 'politifact')
except Exception as e: print(f'  skip politifact: {e}')

# 10. Health misinformation
try:
    ds = load_dataset('Whispering-GPT/fake-news-health', split='train')
    df = ds.to_pandas()
    text_col = next((c for c in ['text','statement','claim','title'] if c in df.columns), df.columns[0])
    label_col = next((c for c in ['label','fake','class'] if c in df.columns), df.columns[-1])
    df['text'] = df[text_col].fillna('').str.strip()
    df['label'] = pd.to_numeric(df[label_col], errors='coerce')
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    add(df, 'health')
except Exception as e: print(f'  skip health: {e}')

print(f'\nFrames loaded: {len(frames)}')

In [ ]:
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

df_all = pd.concat(frames, ignore_index=True)
df_all = df_all.dropna(subset=['text','label'])
df_all = df_all[df_all['text'].str.len() >= 20]
df_all = df_all[df_all['text'].str.contains(r'[a-zA-Z]', regex=True)]
df_all['text'] = df_all['text'].str[:512]
df_all = df_all.drop_duplicates(subset=['text'])
df_all['label'] = df_all['label'].astype(int)
df_all = df_all[df_all['label'].isin([0,1])]

print(f'After cleaning: {len(df_all):,}')
print(df_all.groupby('source')['label'].count().sort_values(ascending=False))

fake = df_all[df_all['label']==1]
real = df_all[df_all['label']==0]
target = min(60000, max(len(fake), len(real)))
fake_b = resample(fake, n_samples=target, random_state=42, replace=len(fake)<target)
real_b = resample(real, n_samples=target, random_state=42, replace=len(real)<target)
df_bal = pd.concat([fake_b, real_b]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Balanced: {len(df_bal):,} (fake={df_bal["label"].sum():,} real={(df_bal["label"]==0).sum():,})')

train_df, temp = train_test_split(df_bal, test_size=0.15, random_state=42, stratify=df_bal['label'])
val_df, test_df = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])
liar_test = df_all[df_all['source']=='liar_test'].copy()
print(f'Train:{len(train_df):,} Val:{len(val_df):,} Test:{len(test_df):,} LIAR-hard:{len(liar_test):,}')

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE_MODEL = 'microsoft/deberta-v3-base'
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)

def to_ds(df):
    ds = Dataset.from_dict({'text': df['text'].tolist(), 'labels': df['label'].tolist()})
    return ds.map(tokenize, batched=True, batch_size=256, remove_columns=['text'])

print('Tokenizing...')
train_ds = to_ds(train_df)
val_ds   = to_ds(val_df)
test_ds  = to_ds(test_df)
train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('Done')

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'REAL', 1:'FAKE'}, label2id={'REAL':0,'FAKE':1}
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

acc_m = evaluate.load('accuracy')
f1_m  = evaluate.load('f1')

def compute_metrics(ep):
    preds = ep[0].argmax(axis=-1)
    return {
        'accuracy': acc_m.compute(predictions=preds, references=ep[1])['accuracy'],
        'f1_macro': f1_m.compute(predictions=preds, references=ep[1], average='macro')['f1']
    }

args = TrainingArguments(
    output_dir='./deberta-factchecker',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1.5e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=False,
    bf16=False,
    gradient_accumulation_steps=2,
    dataloader_num_workers=2,
    logging_steps=100,
    report_to='none',
    save_total_limit=2,
    seed=42,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print('Trainer ready')

In [ ]:
print('Starting training (~55-75 min on T4, keep tab open)...')
result = trainer.train()
print(f'Done. Steps:{result.global_step} Loss:{result.training_loss:.4f}')

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import brier_score_loss
import torch.nn.functional as F
import torch, json, datetime

res = trainer.evaluate(test_ds)
print(f'Test Accuracy: {res["eval_accuracy"]:.4f}  F1: {res["eval_f1_macro"]:.4f}')

out = trainer.predict(test_ds)
y_pred = out.predictions.argmax(axis=-1)
y_true = out.label_ids
print(classification_report(y_true, y_pred, target_names=['Real','Fake']))

probs = F.softmax(torch.tensor(out.predictions), dim=-1).numpy()
brier = brier_score_loss(y_true, probs[:,1])
print(f'Brier score: {brier:.4f}')

if len(liar_test) > 0:
    print('\n--- Hard LIAR benchmark ---')
    ld = to_ds(liar_test)
    ld.set_format('torch')
    lo = trainer.predict(ld)
    print(classification_report(lo.label_ids, lo.predictions.argmax(axis=-1), target_names=['Real','Fake']))

version = {
    'model': MODEL_NAME, 'base': BASE_MODEL,
    'version': datetime.datetime.utcnow().strftime('%Y%m%d_%H%M'),
    'accuracy': round(float(res['eval_accuracy']),4),
    'f1_macro': round(float(res['eval_f1_macro']),4),
    'brier_score': round(float(brier),4),
    'train_samples': len(train_df), 'max_length': MAX_LEN
}
import os
os.makedirs('./deberta-factchecker', exist_ok=True)
with open('./deberta-factchecker/model_version.json','w') as f:
    json.dump(version, f, indent=2)
print(json.dumps(version, indent=2))

In [ ]:
if HF_TOKEN:
    print(f'Pushing to {MODEL_NAME}...')
    trainer.push_to_hub(MODEL_NAME)
    tokenizer.push_to_hub(MODEL_NAME)
    print(f'Live at: https://huggingface.co/{MODEL_NAME}')
else:
    trainer.save_model('./deberta-factchecker')
    tokenizer.save_pretrained('./deberta-factchecker')
    print('Saved locally (no HF token)')

from google.colab import files
files.download('./deberta-factchecker/model_version.json')
print('Downloaded model_version.json - drop into backend/data/')
print(f'Then set env var: DEBERTA_MODEL={MODEL_NAME}')